# Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

##PCA Model

In [ ]:
import os
print(os.listdir("path"))

In [ ]:
!pip install tqdm

In [ ]:
# ✅ Make sure all libraries are imported first
import os
import pandas as pd
import numpy as np
from tqdm import tqdm

# ✅ Define base and subdirectory paths correctly
base_path = "path"
train_dir = os.path.join(base_path, "path")
test_dir = os.path.join(base_path, "path")
metadata_path = os.path.join(base_path, "path")

# Utility function to extract upper triangle
def extract_upper_triangle(file_path):
    matrix = pd.read_csv(file_path, sep='\t', header=None).values
    upper_tri_indices = np.triu_indices_from(matrix, k=1)
    return matrix[upper_tri_indices]

# Process training data
train_vectors = []
train_ids = []

for filename in tqdm(os.listdir(train_dir)):
    if filename.endswith(".tsv"):
        participant_id = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")
        file_path = os.path.join(train_dir, filename)
        vector = extract_upper_triangle(file_path)
        train_vectors.append(vector)
        train_ids.append(participant_id)

# Create DataFrame from vectors and IDs
X_train_raw = pd.DataFrame(train_vectors)
X_train_raw["participant_id"] = train_ids

# Load metadata
metadata = pd.read_csv(metadata_path)
metadata_age = metadata[["participant_id", "age"]]

# Merge features with age from metadata
X_train_raw_with_age = pd.merge(X_train_raw, metadata_age, on="participant_id", how="left")
print(X_train_raw_with_age.head(3))


In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor

X = X_train_raw_with_age.drop(columns=["participant_id", "age"])
y = X_train_raw_with_age["age"]

X_red = PCA(n_components=0.95).fit_transform(X)
print(X_red.shape)

ModelRegr = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1).fit(X_red, y)

print(ModelRegr.score(X_red, y))

In [ ]:
from sklearn.metrics import mean_squared_error
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

def root_mean_squared_error(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
pred_y = ModelRegr.predict(X_red)
print (mean_squared_error(y, pred_y))
print (root_mean_squared_error(y, pred_y))

In [ ]:
# Process Test Data
test_vectors = []
test_ids = []

for filename in tqdm(os.listdir(test_dir)):
    if filename.endswith(".tsv"):
        participant_id = filename.split("_")[0].replace("sub-", "").replace(".tsv", "")
        file_path = os.path.join(test_dir, filename)
        vector = extract_upper_triangle(file_path)
        test_vectors.append(vector)
        test_ids.append(participant_id)

X_test_raw = pd.DataFrame(test_vectors)
X_test_raw["participant_id"] = test_ids

In [ ]:
# Drop participant_id and apply the same PCA transform
X_test = X_test_raw.drop(columns=["participant_id"])
X_test_red = PCA(n_components=0.95).fit(X).transform(X_test)  # ⚠️ Best to reuse PCA fit from train

# To avoid data leakage, re-use the PCA object already fit on X
pca = PCA(n_components=0.95)
X_red = pca.fit_transform(X)
X_test_red = pca.transform(X_test)

In [ ]:
# Predict ages for test set
test_predictions = ModelRegr.predict(X_test_red)

In [ ]:
# Create a DataFrame for submission
submission = pd.DataFrame({
    "participant_id": test_ids,
    "age": test_predictions
})

# Save File path
submission_path = "path"
submission.to_csv(submission_path, index=False)

print("Submission file saved:", submission_path)

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(ModelRegr, X_red, y, cv=5, scoring='neg_mean_squared_error')
rmse_scores = np.sqrt(-cv_scores)
print("Cross-validated RMSE scores:", rmse_scores)
print("Mean RMSE:", np.mean(rmse_scores))

#Hyperparameter Tuning with GridSearchCV

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

# Randomized parameter space
param_dist = {
    'n_estimators': randint(100, 300),
    'max_depth': randint(10, 30),
    'min_samples_split': randint(2, 6),
    'min_samples_leaf': randint(1, 3)
}

rf = RandomForestRegressor(random_state=42, n_jobs=-1)

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=20,  # Try 20 combinations
    scoring='neg_mean_squared_error',
    cv=3,  # Use 3-fold CV instead of 5
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_red, y)

print("Best Params:", random_search.best_params_)
print("Best RMSE:", np.sqrt(-random_search.best_score_))


#Feature Importance Visualization

In [ ]:
import matplotlib.pyplot as plt

# Only works if model is trained on original features (not PCA-reduced)
importances = ModelRegr.feature_importances_

# Sort top 20 features
indices = np.argsort(importances)[-20:][::-1]
plt.figure(figsize=(10, 6))
plt.title("Top 20 Feature Importances")
plt.bar(range(len(indices)), importances[indices])
plt.xlabel("Feature Index")
plt.ylabel("Importance")
plt.show()